# 调用图分析与强连通分量 (SCC) 检测

本笔记本探讨了 UnifyWeaver 的高级代码分析功能：

- **调用图构建** — 从 Prolog 代码构建依赖图
- **SCC 检测** — 发现强连通分量（互递归）
- **模式分析** — 理解递归模式
- **依赖可视化** — 直观呈现谓词之间的关系

## 学习目标

- 理解 UnifyWeaver 如何分析代码结构
- 构建并检查调用图
- 使用 Tarjan 算法检测互递归
- 对代码依赖进行可视化

## 环境配置

加载 UnifyWeaver 及分析模块。

In [ ]:
% 加载初始化
['../init'].

% 加载分析模块
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## 示例 1：简单调用图

从一个简单的谓词开始并构建其调用图。

In [ ]:
% 定义 ancestor 谓词
:- dynamic ancestor/2.
:- dynamic parent/2.

% parent 事实
parent(abraham, isaac).
parent(isaac, jacob).

% ancestor 规则
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### 构建调用图

In [ ]:
% 为 ancestor 构建调用图
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 分析依赖关系

In [ ]:
% 获取 ancestor/2 的所有依赖关系
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% 检查是否自递归
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## 示例 2：互递归检测

现在通过奇偶数示例来检测互递归。

In [ ]:
% 定义互递归谓词
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### 为两个谓词构建调用图

In [ ]:
% 为这两个谓词构建调用图
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 查找强连通分量 (SCC)

In [ ]:
% 重新构建图，因为变量不会在笔记本单元格之间持久保存
build_call_graph([is_even/1, is_odd/1], _Graph),
% 使用 Tarjan 算法查找 SCC
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### 检查 SCC 是否为平凡分量

In [ ]:
% 重新计算衍生值，使该单元格也能独立运行
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% 检查每个 SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## 示例 3：复杂调用图

分析包含多个谓词的更复杂的系统。

In [ ]:
% 定义一个包含多个谓词的小程序
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent 使用 parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling：相同父母，不同孩子
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin：父母是兄弟姐妹
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### 构建完整调用图

In [ ]:
% 为所有谓词构建调用图
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### 查找谓词组

查找包含起始谓词的互递归谓词组。

In [ ]:
% 查找包含 cousin/2 的互递归组
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## 示例 4：模式检测

使用模式匹配器分析递归类型。

In [ ]:
% 定义各种递归模式
:- dynamic count/3.     % 尾递归
:- dynamic factorial/2. % 线性递归
:- dynamic fib/2.       % 树形递归（若被检测到则为线性递归）

% 尾递归计数
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% 线性递归阶乘
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% 斐波那契（可被检测为线性或树形递归）
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### 检测尾递归

In [ ]:
% 检查 count/3 是否为尾递归
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### 检测线性递归

In [ ]:
% 检查 factorial/2 是否为线性递归
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### 统计递归调用次数

In [ ]:
% 统计 fibonacci 中的递归调用次数
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## DOT 格式可视化

生成调用图的 Graphviz DOT 表示。

In [ ]:
% 用于生成 DOT 格式的辅助函数
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% 为奇偶图生成 DOT
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### 保存 DOT 文件

In [ ]:
% 重新构建 DOT 源码，因为变量不会在单元格之间持久保存
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## 练习：分析你自己的代码

尝试定义你自己的谓词并对其进行分析！

In [ ]:
% 在此定义你的谓词
% 然后构建调用图，查找 SCC 并检测模式

% 示例：
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## 总结

在本笔记本中，你学习了：

✅ 如何从 Prolog 代码构建调用图

✅ 如何检测互递归的强连通分量 (SCC)

✅ 如何使用模式匹配器分类递归类型

✅ 如何分析谓词依赖关系

✅ 如何使用 DOT 格式可视化调用图

## 进阶主题

深入分析主题：

- **拓扑排序**：使用 `topological_order/2` 按依赖关系对 SCC 进行排序
- **自定义模式匹配器**：编写专属模式检测谓词
- **累加器模式提取**：使用 `extract_accumulator_pattern/2` 进行详细分析
- **禁用线性递归**：使用 `forbid_linear_recursion/1` 强制采用其他编译策略

## 参考与相关文件

- 第 10 章：Prolog 内省与理论
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`